# Notebook 01 — Lógica Multivalorada aplicada à base de diabetes

Neste notebook, vamos trabalhar com a ideia de que nem toda decisão precisa ser apenas **verdadeira** ou **falsa**.

A lógica multivalorada permite usar mais de dois estados de verdade.  
Para fins didáticos, vamos pensar em três possibilidades principais:

- **0**: evidência ausente;
- **0,5**: evidência intermediária, incerta ou desconhecida;
- **1**: evidência presente.

No contexto da base de diabetes, cada sintoma pode ser tratado como uma evidência.  
Se o sintoma aparece como `Yes`, ele contribui para a suspeita.  
Se aparece como `No`, ele reduz a suspeita.  
Se estiver ausente, ele não será simplesmente tratado como negativo: pode receber um valor intermediário.

## Etapa 1 — Enviar o arquivo da base

Execute a célula abaixo e selecione o arquivo `diabetes_data_upload.csv`.

No Google Colab, esse upload precisa ser feito manualmente sempre que o ambiente for reiniciado.

In [1]:
# ============================================================
# 1. Faça upload do arquivo diabetes_data_upload.csv
# ============================================================

from google.colab import files
import io
import pandas as pd

print("Selecione o arquivo diabetes_data_upload.csv no seu computador.")
uploaded = files.upload()

# Pega automaticamente o primeiro arquivo enviado
nome_arquivo = list(uploaded.keys())[0]

df = pd.read_csv(io.BytesIO(uploaded[nome_arquivo]))

print("Arquivo carregado com sucesso!")
print(f"Nome do arquivo: {nome_arquivo}")
print(f"Quantidade de linhas: {df.shape[0]}")
print(f"Quantidade de colunas: {df.shape[1]}")

df.head()

ModuleNotFoundError: No module named 'google'

## Etapa 2 — Preparar funções auxiliares

Antes de implementar a lógica, vamos criar algumas funções simples.

A função mais importante aqui é a conversão:

- `Yes` vira `1`;
- `No` vira `0`;
- informação ausente fica como `NaN`.

Depois, a lógica multivalorada decidirá como tratar esses casos ausentes.

In [ ]:
# ============================================================
# Bibliotecas e funções auxiliares
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Coluna-alvo da base
TARGET = "class"

# Marcadores clínico-sintomáticos presentes na base
SINTOMAS = [
    "Polyuria",
    "Polydipsia",
    "sudden weight loss",
    "weakness",
    "Polyphagia",
    "Genital thrush",
    "visual blurring",
    "Itching",
    "Irritability",
    "delayed healing",
    "partial paresis",
    "muscle stiffness",
    "Alopecia",
    "Obesity",
]

# Sintomas escolhidos como centrais para simular lacunas ou conflito
# A escolha é didática: são sintomas que aparecem com relevância clínica
# e ajudam a mostrar o comportamento das lógicas.
SINTOMAS_CENTRAIS = [
    "Polyuria",
    "Polydipsia",
    "sudden weight loss",
    "weakness",
    "Polyphagia",
    "partial paresis",
]

# Pesos heurísticos.
# Eles não foram aprendidos por machine learning.
# São apenas uma forma didática de dizer:
# "alguns sintomas pesam mais na suspeita do que outros".
PESOS = {
    "Polyuria": 3.0,
    "Polydipsia": 3.0,
    "sudden weight loss": 2.0,
    "weakness": 1.0,
    "Polyphagia": 1.5,
    "Genital thrush": 0.5,
    "visual blurring": 1.0,
    "Itching": 0.25,
    "Irritability": 1.0,
    "delayed healing": 0.75,
    "partial paresis": 2.0,
    "muscle stiffness": 0.5,
    "Alopecia": 0.25,
    "Obesity": 0.25,
}

def sim_nao_para_numero(valor):
    '''
    Converte respostas categóricas da base para números.

    Yes -> 1
    No  -> 0
    NaN -> NaN

    Essa conversão permite que a lógica opere sobre evidências.
    '''
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip().lower()

    if valor == "yes":
        return 1.0
    elif valor == "no":
        return 0.0
    else:
        return np.nan

def pertinencia_idade(idade):
    '''
    Transforma idade em uma evidência gradual simples.

    Menos de 30 anos: pertinência 0
    Entre 30 e 60 anos: crescimento linear
    Acima de 60 anos: pertinência 1

    Não é uma regra médica definitiva.
    É apenas uma simplificação didática para mostrar como
    variáveis numéricas podem entrar no raciocínio lógico.
    '''
    if pd.isna(idade):
        return 0.5

    if idade < 30:
        return 0.0
    elif idade > 60:
        return 1.0
    else:
        return (idade - 30) / 30

def classe_para_binario(valor):
    '''
    Converte a classe original da base para comparação simples.

    Positive -> 1
    Negative -> 0
    '''
    return 1 if valor == "Positive" else 0

def mostrar_distribuicao_decisoes(resultado, coluna="decisao", titulo="Distribuição das decisões"):
    '''
    Plota um gráfico simples com a distribuição das saídas produzidas pela lógica.
    '''
    contagem = resultado[coluna].value_counts()

    plt.figure(figsize=(8, 4))
    barras = plt.bar(contagem.index.astype(str), contagem.values)
    plt.title(titulo)
    plt.xlabel("Saída produzida pela lógica")
    plt.ylabel("Quantidade de registros")
    plt.xticks(rotation=25, ha="right")

    for barra in barras:
        altura = barra.get_height()
        plt.text(
            barra.get_x() + barra.get_width()/2,
            altura,
            str(int(altura)),
            ha="center",
            va="bottom"
        )

    plt.tight_layout()
    plt.show()

def resumo_simples(resultado, coluna_decisao="decisao"):
    '''
    Gera um resumo didático da saída da lógica.

    A ideia não é tratar a lógica como um classificador tradicional,
    mas observar como ela decide, quando se abstém e como distribui
    suas respostas.
    '''
    total = len(resultado)

    resumo = resultado[coluna_decisao].value_counts().reset_index()
    resumo.columns = ["decisao", "quantidade"]
    resumo["percentual"] = 100 * resumo["quantidade"] / total

    return resumo

## Etapa 3 — Conhecer a base

Antes de aplicar a lógica, vamos olhar a estrutura do conjunto de dados.

Isso ajuda a reforçar um ponto importante: esta base não contém exames laboratoriais, mas sim marcadores clínico-sintomáticos.

In [ ]:
# ============================================================
# Olhando a base antes de aplicar qualquer lógica
# ============================================================

print("Dimensão da base:", df.shape)
print("\nColunas disponíveis:")
print(df.columns.tolist())

print("\nDistribuição da classe:")
display(df[TARGET].value_counts().reset_index().rename(columns={"index": "classe", TARGET: "quantidade"}))

print("\nQuantidade de valores ausentes por coluna:")
display(df.isna().sum().reset_index().rename(columns={"index": "coluna", 0: "ausentes"}))

print("\nResumo da idade:")
display(df["Age"].describe())

## Etapa 4 — Ideia da lógica multivalorada

Neste exemplo, cada paciente recebe um escore calculado a partir dos sintomas.

A diferença para uma lógica binária simples é que podemos usar um valor intermediário quando a informação está ausente.

A regra didática será:

- se houver muitas lacunas centrais, a saída será **indeterminado**;
- se o escore for alto, a saída será **alto**;
- se o escore for intermediário, a saída será **moderado**;
- se o escore for baixo, a saída será **baixo**.

Assim, a lógica não força toda situação para apenas duas respostas.

In [ ]:
def decisao_multivalorada(linha):
    valores = []
    pesos = []

    for sintoma in SINTOMAS:
        valor = sim_nao_para_numero(linha[sintoma])

        # Aqui está a essência multivalorada:
        # quando não sabemos, usamos um estado intermediário.
        if pd.isna(valor):
            valor = 0.5

        valores.append(valor)
        pesos.append(PESOS[sintoma])

    # A idade entra como uma evidência gradual complementar.
    valores.append(pertinencia_idade(linha["Age"]))
    pesos.append(0.75)

    escore = np.average(valores, weights=pesos)

    lacunas_centrais = sum(pd.isna(linha[coluna]) for coluna in SINTOMAS_CENTRAIS)

    if lacunas_centrais >= 3:
        decisao = "indeterminado"
    elif escore >= 0.60:
        decisao = "alto"
    elif escore >= 0.35:
        decisao = "moderado"
    else:
        decisao = "baixo"

    return pd.Series({
        "escore_multivalorado": escore,
        "lacunas_centrais": lacunas_centrais,
        "decisao": decisao
    })

def aplicar_logica_multivalorada(base):
    resultado = base.copy()
    saidas = resultado.apply(decisao_multivalorada, axis=1)
    resultado = pd.concat([resultado, saidas], axis=1)
    resultado["classe_binaria"] = resultado[TARGET].apply(lambda x: 1 if x == "Positive" else 0)
    return resultado

## Etapa 5 — Aplicar a lógica na base original

Agora aplicamos a lógica multivalorada em todos os registros da base.

Observe principalmente a coluna `decisao`.

In [ ]:
resultado_original = aplicar_logica_multivalorada(df)

display(resultado_original[[
    "Age", "Gender", "Polyuria", "Polydipsia",
    "sudden weight loss", "weakness",
    "class", "escore_multivalorado",
    "lacunas_centrais", "decisao"
]].head(10))

display(resumo_simples(resultado_original))
mostrar_distribuicao_decisoes(
    resultado_original,
    titulo="Lógica multivalorada — base original"
)

## Etapa 6 — Simular dados incompletos

A lógica multivalorada é interessante quando queremos representar um estado intermediário.

Vamos apagar parte dos sintomas centrais e observar como a decisão muda.

In [ ]:
# ============================================================
# Criando uma cópia da base com lacunas simuladas
# ============================================================

def criar_base_incompleta(base, taxa_lacunas=0.20, semente=42):
    '''
    Cria uma versão da base com alguns sintomas centrais apagados.

    Isso simula uma situação comum em saúde:
    nem toda informação foi registrada, perguntada ou confirmada.
    '''
    base_incompleta = base.copy()
    rng = np.random.default_rng(semente)

    for coluna in SINTOMAS_CENTRAIS:
        mascara = rng.random(len(base_incompleta)) < taxa_lacunas
        base_incompleta.loc[mascara, coluna] = np.nan

    return base_incompleta

df_incompleta = criar_base_incompleta(df, taxa_lacunas=0.20)

print("Base original:")
display(df[SINTOMAS_CENTRAIS].head())

print("Base com lacunas simuladas:")
display(df_incompleta[SINTOMAS_CENTRAIS].head())

print("Valores ausentes após simulação:")
display(df_incompleta[SINTOMAS_CENTRAIS].isna().sum())

In [ ]:
resultado_incompleto = aplicar_logica_multivalorada(df_incompleta)

display(resultado_incompleto[[
    "Age", "Gender", "Polyuria", "Polydipsia",
    "sudden weight loss", "weakness",
    "class", "escore_multivalorado",
    "lacunas_centrais", "decisao"
]].head(10))

display(resumo_simples(resultado_incompleto))
mostrar_distribuicao_decisoes(
    resultado_incompleto,
    titulo="Lógica multivalorada — base com lacunas"
)

## Etapa 7 — Reflexão

A lógica multivalorada ajuda quando a decisão não deve ser reduzida a sim/não.

No caso deste experimento, ela permite produzir saídas como:

- `baixo`;
- `moderado`;
- `alto`;
- `indeterminado`.

Isso é útil em sistemas de apoio à decisão porque a saída pode expressar níveis intermediários de suspeita, sem fingir que toda evidência clínica é perfeitamente binária.

In [ ]:
resultado_original.to_csv("resultado_multivalorada_original.csv", index=False)
resultado_incompleto.to_csv("resultado_multivalorada_incompleta.csv", index=False)

print("Arquivos exportados:")
print("- resultado_multivalorada_original.csv")
print("- resultado_multivalorada_incompleta.csv")